In [1]:
# run_dinov2_dump_baseline_vs_zero_regs.py
# Dumps DINOv2 (CLS + reg tokens) for baseline and zeroed registers, with lightweight progress prints.

import json, time
from pathlib import Path
from contextlib import contextmanager

import torch
import timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from timm.data import resolve_data_config, create_transform

/home/ubuntu/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path("/lambda/nfs/neel/Research")
SUBSET = "coco_caption_5k"  # or "vqa_v2_balanced_5k"
MODEL_NAME = "vit_base_patch14_reg4_dinov2"

BATCH_SIZE = 32
NUM_WORKERS = 6
SAVE_DTYPE = torch.float16
LOG_EVERY_N_BATCHES = 20

META_PATH = ROOT / "subsets" / SUBSET / "metadata.jsonl"
OUT_DIR = ROOT / "runs" / "dinov2" / SUBSET / "dinov2_dumps" / MODEL_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class MetaImages(Dataset):
    def __init__(self, meta_path: Path, transform):
        self.rows = []
        with meta_path.open("r") as f:
            for line in f:
                line = line.strip()
                if line:
                    self.rows.append(json.loads(line))
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        image_id = str(r.get("image_id", r.get("id", idx)))
        img_path = r.get("image_file") or r.get("image_path") or r.get("path")
        if img_path is None:
            raise KeyError("metadata row missing image_file/image_path/path")

        p = Path(img_path)
        if not p.is_absolute():
            p = ROOT / p

        pil = Image.open(p).convert("RGB")
        x = self.transform(pil)  # (3,H,W) normalized to model cfg
        return image_id, str(p), x

In [4]:
def collate(batch):
    ids, paths, xs = zip(*batch)
    return list(ids), list(paths), torch.stack(xs, dim=0)

In [5]:
@contextmanager
def zero_regs(vit):
    saved = vit.reg_token.detach().clone()
    try:
        vit.reg_token.data.zero_()
        yield
    finally:
        vit.reg_token.data.copy_(saved)

In [6]:
@torch.inference_mode()
def forward_cls_regs(vit, images_bchw):
    R = vit.reg_token.shape[1]
    x = vit.patch_embed(images_bchw)  # (B,N,D)

    if hasattr(vit, "_pos_embed"):
        x = vit._pos_embed(x)         # (B,1+R+N,D)
    else:
        B = x.size(0)
        cls = vit.cls_token.expand(B, -1, -1)
        reg = vit.reg_token.expand(B, -1, -1)
        x = torch.cat([cls, reg, x], dim=1)

        pos = vit.pos_embed
        if pos.shape[1] == (x.shape[1] - (1 + R)):         # patch-only
            x[:, 1+R:, :] = x[:, 1+R:, :] + pos
        elif pos.shape[1] == x.shape[1]:                   # full
            x = x + pos
        elif pos.shape[1] == (x.shape[1] - R):             # cls+patch
            x[:, 0:1, :] = x[:, 0:1, :] + pos[:, 0:1, :]
            x[:, 1+R:, :] = x[:, 1+R:, :] + pos[:, 1:, :]
        else:
            raise RuntimeError(f"Unhandled pos_embed {tuple(pos.shape)} vs tokens {tuple(x.shape)}")

        if hasattr(vit, "pos_drop"):
            x = vit.pos_drop(x)

    if hasattr(vit, "norm_pre") and vit.norm_pre is not None:
        x = vit.norm_pre(x)

    for blk in vit.blocks:
        x = blk(x)

    if hasattr(vit, "norm") and vit.norm is not None:
        x = vit.norm(x)

    return x[:, 0, :], x[:, 1:1+R, :]  # cls (B,D), regs (B,R,D)

In [7]:
def dump_condition(vit, dataset, condition: str):
    cond_dir = OUT_DIR / condition
    cond_dir.mkdir(parents=True, exist_ok=True)
    index_path = OUT_DIR / f"index_{condition}.jsonl"

    dl = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate,
    )

    total = len(dataset)
    done = 0
    t0 = time.time()

    with index_path.open("w") as f:
        for bi, (ids, paths, x) in enumerate(dl):
            x = x.to(DEVICE, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                cls, regs = forward_cls_regs(vit, x)

            cls = cls.to("cpu", dtype=SAVE_DTYPE)
            regs = regs.to("cpu", dtype=SAVE_DTYPE)

            for i, (image_id, image_path) in enumerate(zip(ids, paths)):
                out_path = cond_dir / f"{image_id}.pt"
                torch.save({"cls": cls[i], "regs": regs[i]}, out_path)  # cls:(D,), regs:(R,D)

                f.write(json.dumps({
                    "image_id": image_id,
                    "image_file": image_path,
                    "condition": condition,
                    "model": MODEL_NAME,
                    "dump_path": str(out_path),
                    "dtype": "float16" if SAVE_DTYPE == torch.float16 else str(SAVE_DTYPE),
                }) + "\n")

            done += len(ids)

            if (bi + 1) % LOG_EVERY_N_BATCHES == 0 or done == total:
                dt = time.time() - t0
                ips = done / dt if dt > 0 else 0.0
                eta = (total - done) / ips if ips > 0 else float("inf")
                print(f"[{condition}] {done}/{total} ({done/total:.1%}) | {ips:.1f} img/s | ETA {eta/60:.1f} min")

    print(f"Done: {condition} -> {cond_dir}")
    print(f"Index: {index_path}")

In [8]:
def main():
    vit = timm.create_model(MODEL_NAME, pretrained=True).eval().to(DEVICE)
    assert hasattr(vit, "reg_token"), "Pick a *_reg4_dinov2 model."

    cfg = resolve_data_config(vit.pretrained_cfg, model=vit)
    transform = create_transform(**cfg)

    ds = MetaImages(META_PATH, transform)

    dump_condition(vit, ds, "baseline")
    with zero_regs(vit):
        dump_condition(vit, ds, "zero_regs")

In [9]:
if __name__ == "__main__":
    main()

[baseline] 640/5000 (12.8%) | 23.1 img/s | ETA 3.1 min
[baseline] 1280/5000 (25.6%) | 30.2 img/s | ETA 2.1 min
[baseline] 1920/5000 (38.4%) | 33.4 img/s | ETA 1.5 min
[baseline] 2560/5000 (51.2%) | 34.3 img/s | ETA 1.2 min
[baseline] 3200/5000 (64.0%) | 35.1 img/s | ETA 0.9 min
[baseline] 3840/5000 (76.8%) | 35.6 img/s | ETA 0.5 min
[baseline] 4480/5000 (89.6%) | 35.7 img/s | ETA 0.2 min
[baseline] 5000/5000 (100.0%) | 35.3 img/s | ETA 0.0 min
Done: baseline -> /lambda/nfs/neel/Research/runs/dinov2/coco_caption_5k/dinov2_dumps/vit_base_patch14_reg4_dinov2/baseline
Index: /lambda/nfs/neel/Research/runs/dinov2/coco_caption_5k/dinov2_dumps/vit_base_patch14_reg4_dinov2/index_baseline.jsonl
[zero_regs] 640/5000 (12.8%) | 37.3 img/s | ETA 1.9 min
[zero_regs] 1280/5000 (25.6%) | 38.9 img/s | ETA 1.6 min
[zero_regs] 1920/5000 (38.4%) | 38.1 img/s | ETA 1.3 min
[zero_regs] 2560/5000 (51.2%) | 38.1 img/s | ETA 1.1 min
[zero_regs] 3200/5000 (64.0%) | 38.1 img/s | ETA 0.8 min
[zero_regs] 3840/5000